# <b>Vehicle ID Mapping (Dilax -> APC)</b>

This notebook is a continuation of `linking.ipynb`, where we explored how to link the APC raw data with matched_stops. Since the two systems use different vehicle ID systems, we cannot match them directly by ID. Instead, we link the two datasets based on **time** and **GPS coordinates**.

In `linking.ipynb` we tested several tolerance combinations on the full dataset. The combination **±60 seconds** and **±0.0020 degrees** gave the highest share of unique 1-to-1 matches (84%).

In this notebook we apply that combination, keep only the rows with **exactly one APC candidate** (skipping rows with 2+ candidates to stay on the safe side), and use the unique matches to derive a Dilax → APC vehicle ID mapping.

## Setup: Imports And Loading Data

In [2]:
import pandas as pd
import numpy as np

# === Load ===
apc           = pd.read_csv('../data/raw/apc.csv', sep=';', encoding='latin-1')
matched_stops = pd.read_csv('../data/raw/matched_stops.csv', sep=';', decimal=',', encoding='latin-1')

apc['arrival_dt']           = pd.to_datetime(apc['arrival'], format='%d/%m/%Y %H:%M:%S')
matched_stops['arrival_dt'] = pd.to_datetime(matched_stops['arrivalTime'], format='%m-%d-%Y %I:%M:%S %p')

print(f"apc rows:           {len(apc):,}")
print(f"matched_stops rows: {len(matched_stops):,}")

apc rows:           1,598,279
matched_stops rows: 56,165


## Step 1 - Match matched_stops with APC

For each row in matched_stops, we search APC for rows that fall within ±60 seconds and ±0.0020 degrees of the matched_stops time and position. We record how many candidates were found per row, and store the APC vehicle ID when there is exactly one candidate.

To make this fast, we sort APC by time once and use binary search (`searchsorted`) to find the relevant time window for each matched_stops row.

In [3]:
time_tol_sec = 60
gps_tol      = 0.0020

apc_sorted = apc.sort_values('arrival_dt').reset_index(drop=True)
apc_times    = apc_sorted['arrival_dt'].values
apc_lats     = apc_sorted['latitude'].values
apc_lons     = apc_sorted['longitude'].values
apc_vehicles = apc_sorted['vehicleCode'].values

n_matches      = np.zeros(len(matched_stops), dtype=int)
matched_apc_v  = np.full(len(matched_stops), -1, dtype=int)

for i in range(len(matched_stops)):
    ms_row = matched_stops.iloc[i]
    t = ms_row['arrival_dt'].to_datetime64()
    
    left  = np.searchsorted(apc_times, t - np.timedelta64(time_tol_sec, 's'))
    right = np.searchsorted(apc_times, t + np.timedelta64(time_tol_sec, 's'))
    
    if left == right:
        continue
    
    lat_ok = np.abs(apc_lats[left:right] - ms_row['latitude'])  <= gps_tol
    lon_ok = np.abs(apc_lons[left:right] - ms_row['longitude']) <= gps_tol
    in_window = lat_ok & lon_ok
    n = int(in_window.sum())
    n_matches[i] = n
    
    if n == 1:
        matched_apc_v[i] = apc_vehicles[left:right][in_window][0]

matched_stops['n_matches']   = n_matches
matched_stops['apc_vehicle'] = matched_apc_v

print()
print(f"0 matches:        {(n_matches == 0).sum():,}")
print(f"1 match (unique): {(n_matches == 1).sum():,}")
print(f"2+ matches:       {(n_matches >= 2).sum():,}")


0 matches:        4,028
1 match (unique): 46,985
2+ matches:       5,152


## Step 2 - Keep only unique matches

We discard rows with 0 or 2+ matches. Rows with 2+ matches are ambiguous: multiple APC rows fall inside the time and GPS window, often because two different buses passed nearby within a minute. To avoid contaminating the mapping with wrong assignments, we only keep rows where the match is unique.

In [4]:
unique = matched_stops[matched_stops['n_matches'] == 1].copy()
print(f"\nKept {len(unique):,} unique matches")
print(f"\nUnique Dilax vehicles in linked data: {unique['vehicleNumber'].nunique()}")


Kept 46,985 unique matches

Unique Dilax vehicles in linked data: 23


## Step 3 - Build the vehicle ID mapping table

For each Dilax vehicle, we look at all its unique matches and find the APC vehicle that appears most often. We also report:

- `n_links`: how many unique matches we have for this Dilax vehicle
- `confidence`: the share of links pointing to the most common APC vehicle
- `distinct_apc`: how many different APC vehicles appeared at all

A confidence near 1.0 and a small `distinct_apc` indicates a clean, reliable mapping. Lower confidence or many distinct APC vehicles suggests the GPS+time matching occasionally picked up the wrong bus (e.g. a neighbouring bus passing the same stop within the time window).

In [5]:
mapping_rows = []
for dilax_v, grp in unique.groupby('vehicleNumber'):
    counts = grp['apc_vehicle'].value_counts()
    mapping_rows.append({
        'dilax_vehicle': dilax_v,
        'apc_vehicle':   counts.index[0],
        'n_links':       int(counts.sum()),
        'confidence':    counts.iloc[0] / counts.sum(),
        'distinct_apc':  len(counts),
    })

mapping = pd.DataFrame(mapping_rows).sort_values('dilax_vehicle').reset_index(drop=True)
print(mapping.to_string(index=False))

 dilax_vehicle  apc_vehicle  n_links  confidence  distinct_apc
            90         4288       28    0.964286             2
           106         4259      251    0.980080             4
           117         4270        4    0.500000             2
           133         4205     3649    0.976980            15
           138         4197    17028    0.982265            41
           204         4213      700    0.964286             9
           216         4222     1981    0.987885             7
           219         4226     3003    0.973360            13
           222         4212      321    1.000000             1
           224         4240     5334    0.985189            17
           240         4227     5951    0.975970            22
           244         4267       15    0.866667             2
           249         4317      186    0.973118             3
           262         4241     3380    0.989645            11
           264         4246     2068    0.990812       

## Step 4 - Save results

We save two files:

- `matched_stops_linked.csv`: the unique-match rows with both Dilax and APC 
  vehicle IDs, useful for further per-row analysis.
- `vehicle_mapping.csv`: the 23-row mapping table that is the main output of 
  this notebook.

In [6]:
unique[['arrival_dt', 'vehicleNumber', 'apc_vehicle', 'pointShortLabel', 'stopLabel', 'tripNumber']].to_csv(
    '../data/processed/matched_stops_linked.csv', sep=';', index=False, encoding='latin-1')
mapping.to_csv('../data/processed/vehicle_mapping.csv', sep=';', decimal=',', index=False, encoding='latin-1')

print("\nSaved:")
print("  ../data/processed/matched_stops_linked.csv")
print("  ../data/processed/vehicle_mapping.csv")


Saved:
  ../data/processed/matched_stops_linked.csv
  ../data/processed/vehicle_mapping.csv
